# 43008: Reinforcement Learning - WOW Naval Fleet: MAPPO vs PQN

## Stage 0: Gunnery

[Imitation](StageP-Imitation.ipynb) · **Stage 0** · [Stage 1](Stage1-Capture.ipynb) · [Stage 2](Stage2-Defend.ipynb) · [Stage 3](Stage3-Duel.ipynb) · [Stage 4](Stage4-King-of-the-Hill.ipynb) · [Stage 5](Stage5-Archipelago.ipynb) · [Stage 6](Stage6-Domination.ipynb) · [Stage 7](Stage7-Open.ipynb) · [Stage 8](Stage8-Archipelago-9v9.ipynb) · [MAPPO vs PQN](Versus-MAPPO-vs-PQN.ipynb) · [the full walkthrough](WOW-MAPPO-Simple.ipynb)

| Stage | Battle | Fleet | Opponent | Promotion requirement |
|---|---|---|---|---|
| 0 - Gunnery | `gunnery` | 1 battleship vs 1 cruiser | Passive target (stop, hold fire) | target sunk within 150 s in ≥70% of the last 50 battles - or, NOT passed, after 80,000 timesteps on it (its cap), it moves on anyway |

### What the fleet learns here

The first lesson is the most basic one: **destroy a ship**. Our battleship starts 7 km from an enemy cruiser that neither moves nor shoots - Python flies it with "stop, hold fire". The circle reward is off; damage and the kill pay, and the **win pays only for a kill within 150 s** - at 150 s without one the battle ends as a loss (against a target that never scores, the game itself would call every battle won, however slow).

A battle is *passed* when the target is sunk within 150 seconds - just sinking it eventually is too easy (random play does it), so the stage asks for doing it quickly: picking the target, keeping the guns firing, closing in and using torpedoes.

Measured (32 battles each): random play sinks the target within 150 s in 52% of battles, the script "close in, fire at will, torpedoes" 76%.

### The ships

| Class | Our fleet | Enemy fleet |
|---|---|---|
| Battleship (BB) | 1 | 0 |
| Cruiser (CA) | 0 | 1 |
| Destroyer (DD) | 0 | 0 |
| Submarine (SS) | 0 | 0 |
| **Total** | **1** | **1** |

The enemy is a passive target: Python orders it to stop and hold fire.

| Class | | What it does |
|---|---|---|
| **Battleship** | BB | the biggest guns and the thickest armour; slow, long range - wins duels, anchors the line |
| **Cruiser** | CA | medium guns that fire fast, radar / hydro / smoke support; the all-rounder |
| **Destroyer** | DD | the fastest, torpedoes and smoke, hardest to spot - the circle capper and scout |
| **Submarine** | SS | dives under water, torpedoes, very hard to spot or hit while submerged |

### One model per method, through every stage

The stage notebooks train **two runs** - the MAPPO fleet (`RUN_NAME`) and the PQN fleet (`PQN_RUN_NAME`, `RUN_NAME + "_pqn"`) - each through every stage. A session continues each run's latest checkpoint, which carries everything learned so far (networks, optimisers, the league, the timestep count, the stage) and its battle log, from which the promotion window continues where the last session stopped.

* **Training is continuous.** Not passed yet? **Run this notebook again** - each session continues the last one (yours or a teammate's), from `models/model_latest.pt`, refreshed every `save_interval`. `KEEP_GOING = True` carries on into the next stages in the same session instead of stopping at the promotion.
* `TRAIN_MAPPO` / `TRAIN_PQN` (configuration cell) switch either method's training on or off for this session. A run that is on another stage is not trained here, only evaluated for the comparison.
* **Train this stage again** - after a run passed it, reached its cap, or is on another stage: set `FORCE_STAGE = True` (and `TRAIN_MAPPO` / `TRAIN_PQN` to choose the method) and run the notebook. The run comes back to this stage and trains it with a fresh promotion window and cap, until it passes again, reaches the cap, or `n_timesteps` is done; then it is on this stage - or the next one, if it passed - and the notebooks carry on from there.
* Passed? That run stops and saves, and moves to stage 1: open [Stage 1](Stage1-Capture.ipynb).
* No model yet? Both start from the imitation checkpoint of the [Imitation notebook](StageP-Imitation.ipynb) - run that one first.

### Two methods, the same fleet

Every stage notebook trains **two** runs on the same battles - one per method - and compares them:

| | **MAPPO** (`simple_mappo/mappo.py`) | **PQN** (`simple_mappo/pqn.py`) |
|---|---|---|
| Family | policy gradient (PPO, Week 10) | value-based Q-learning - the 2025 successor of DQN (Week 9): **PQN**, the Parallelised Q-Network (Gallici et al., ICLR 2025), which beat QMIX and MAPPO on SMAX (StarCraft multi-agent) |
| What the captain's head scores are | logits: sample an option | Q-values (dueling: V + A): sample an option by softmax(Q / 0.1) - Boltzmann exploration (`exploration="epsilon"`: the best option, epsilon-greedy) |
| Learns from | the battles it just played (on-policy) | **also the battles it just played**: a rollout of 128 decisions from every battle, 4 epochs of minibatches, then thrown away - no replay buffer, so self-play opponents never go stale |
| Target | GAE advantages from a centralised critic | **Q(λ) returns**, λ = 0.85: R_t = r_t + γ[λR_t+1 + (1−λ) max Q(s_t+1)], from the network as it was at the rollout; no separate target network - LayerNorm in the Q-network keeps it stable |
| Six action heads | one probability per head | one Q per head, every head trained to the same target (branching DQN, BDQ) |
| Fleet commander | PPO at a 10 s clock | Q-learning on (ship, order) at a 10 s clock (h-DQN, intra-option) |
| Many ships | shared weights, centralised critic | shared weights; **PQN-VDN**: the fleet's Q is the sum of its ships' Q-values, trained on the fleet's reward |
| Shared | the captain network (transformer + GRU + pointer heads), the commander, the rewards, the curriculum, self-play, the imitation start | ← same |

Each run climbs the curriculum on its own (`runs_simple/fleet` and `runs_simple/fleet_pqn`), so a stage notebook only trains a run that is on its stage; the other one is still evaluated here, for the comparison.

### MAPPO in one picture

| Piece | What it does | Where it comes from | Course link |
|---|---|---|---|
| **Captain** (every ship, one shared network) | its own view → 6 action heads; a transformer over ships/circles/islands, a GRU memory, pointer heads for *which target* and *which circle* | AlphaStar, OpenAI Five | W10-C custom feature extractor; frame stacking → memory |
| **Fleet commander** | every 10 s: an order per ship - free / engage / hold circle k | Honor of Kings' hierarchical macro strategy | the same PPO at a slower clock |
| **Centralised critic** | sees the true battle (training only); one value per reward group + win probability | MAPPO, Honor of Kings multi-head value | W10-A/B actor-critic |
| **Update** | PPO with dual clip, value normalisation and clipping | MAPPO, Honor of Kings | W10-B PPO |
| **Warm start** | the captain first copies the Elite rule AI (StageP), then a fading KL penalty keeps it close | AlphaStar | W9-A supervised training loop |
| **Self-play & league** | from stage 3: mirror battles (the fleet on **both** sides, **both** learning), its **champion** (replaced only when beaten in 55% of 40 battles), past copies (PFSP), and the **other method's fleet** (cross-play), with Elo ratings; the rule AI stays as the anchor promotion is judged against | OpenAI Five, AlphaGo Zero, AlphaStar | W9-C/D frozen target network |
| **Curriculum** | 8 stages, each passed at ≥70% of its own goal; earlier stages are replayed 10-15% of the time | TiZero, OpenAI Five | - |

## Setup

On SageMaker, run this from the unzipped bundle (`<folder>/Training/`). On your computer, open it from the repository's `Training/` folder. Either way the battles are the **real Unity game** (the headless player in `Builds/NavalTrainer`, built from this project's `Assets/`), not a simplified copy.

In [ ]:
import os, time, zipfile
# SageMaker Studio (/home/sagemaker-user) or a SageMaker notebook instance (/home/ec2-user/SageMaker)
ON_SAGEMAKER = any(os.path.isdir(p) for p in ("/home/sagemaker-user", "/home/ec2-user/SageMaker"))
home = os.path.expanduser("~/SageMaker") if os.path.isdir(os.path.expanduser("~/SageMaker")) else os.path.expanduser("~")
bundle = os.path.join(home, "wow_sagemaker.zip")
if ON_SAGEMAKER and os.path.basename(os.getcwd()) != "Training":
    if not os.path.isdir(os.path.join(home, "wow")):
        assert os.path.exists(bundle), f"upload wow_sagemaker.zip to {home} first"
        zipfile.ZipFile(bundle).extractall(os.path.join(home, "wow"))
    %cd {home}/wow/Training

# Uploading a new wow_sagemaker.zip does not replace the unzipped folder: check which version runs here
running = open("BUNDLE_VERSION").read().strip() if os.path.exists("BUNDLE_VERSION") else "an older one"
if ON_SAGEMAKER and os.path.exists(bundle):
    with zipfile.ZipFile(bundle) as z:
        uploaded = z.read("Training/BUNDLE_VERSION").decode().strip() if "Training/BUNDLE_VERSION" in z.namelist() else "unknown"
    if uploaded != running:
        print(f"*** {bundle} is version {uploaded}, but this notebook runs version {running}. Unzip it into a new folder "
              f"and open this notebook from there. ***")
print(os.getcwd(), "| bundle", running, "| glibc", os.confstr("CS_GNU_LIBC_VERSION"))

In [ ]:
# install only what is missing (PyTorch as the CPU build)
import importlib.util, shutil
missing = [pkg for mod, pkg in (("numpy", "numpy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("PIL", "pillow"))
           if importlib.util.find_spec(mod) is None]
if shutil.which("ffmpeg") is None and importlib.util.find_spec("imageio_ffmpeg") is None:
    missing.append("imageio-ffmpeg")      # an ffmpeg: MP4 replays to keep, and GIFs of a teammate's MP4s
if missing:
    !pip install -q {" ".join(missing)}
if importlib.util.find_spec("torch") is None:
    !pip install -q torch --index-url https://download.pytorch.org/whl/cpu

In [ ]:
import os
from collections import OrderedDict

import numpy as np
import pandas as pd

from simple_mappo import (NavalEnv, MAPPO, FleetPQN, Monitor, CurriculumCallback, SaveOnIntervalCallback,
                          BestModelCallback, VideoCallback, TeamStorage, TeamSyncCallback, DEFAULT_CURRICULUM,
                          open_run, training_status, evaluate, plot_results, plot_curriculum, plot_reward_terms,
                          plot_progress, plot_comparison, compare_stages, show_videos, find_videos, record_battle)
from simple_mappo.utils import video_ext

## Team checkpoints

Use the **same `RUN_NAME` in every notebook** (and as your teammates): that is the one line of training the stages hand over along - one per method. Only one person can train a run at a time.

In [ ]:
YOUR_NAME = "your-name"              # shown to your teammates
RUN_NAME = "fleet"                   # the MAPPO run - the same in every stage notebook (and StageP)
if ON_SAGEMAKER:
    from simple_mappo.aws import team_storage_uri
    TEAM_STORAGE = team_storage_uri()     # s3://sagemaker-<region>-<account>/wow-mappo, or "s3://<team-bucket>/wow-mappo"
else:
    TEAM_STORAGE = None              # e.g. "s3://<team-bucket>/wow-mappo"; None trains on this machine only

storage = TeamStorage(TEAM_STORAGE, RUN_NAME, user=YOUR_NAME) if TEAM_STORAGE else None
run_dir = f"runs_simple/{RUN_NAME}"
log_dir, models_dir, video_dir = (os.path.join(run_dir, d) for d in ("logs", "models", "videos"))
os.makedirs(models_dir, exist_ok=True)
if storage:
    storage.show_status()

In [ ]:
PQN_RUN_NAME = RUN_NAME + "_pqn"     # the PQN run: its own checkpoints, logs and curriculum stage
pqn_storage = TeamStorage(TEAM_STORAGE, PQN_RUN_NAME, user=YOUR_NAME) if TEAM_STORAGE else None
pqn_run_dir = f"runs_simple/{PQN_RUN_NAME}"
pqn_log_dir, pqn_models_dir, pqn_video_dir = (os.path.join(pqn_run_dir, d) for d in ("logs", "models", "videos"))
os.makedirs(pqn_models_dir, exist_ok=True)
if pqn_storage:
    pqn_storage.show_status()

## Configuration

Each method's settings are **identical in every notebook**. They only shape a brand-new model; a continued model keeps the settings it was created with. `n_timesteps` is this session's training, per method; rerun the notebook to continue.

In [ ]:
STAGE = 0
FORCE_STAGE = False                  # True: train this stage (again) even if a run passed it, reached its cap
                                     # or is on another stage - with a fresh promotion window and cap
KEEP_GOING = False                   # True: when this stage is passed, carry on into the next stages
                                     # in this same session instead of stopping (continuous training)
TRAIN_MAPPO = True                   # train the MAPPO run in this session
TRAIN_PQN = True                    # train the PQN run in this session

config = OrderedDict([
    ('n_envs', max(1, min(16, os.cpu_count() // 2))),     # battles in parallel, one Unity process each
    ('decision_period', 1.0),          # game seconds between decisions
    ('sim_dt', 0.08),                  # game seconds per simulated frame (fastest stable)
    ('n_timesteps', 80_000),          # this session's training, per method (rises stage by stage)
])
mappo_settings = dict(learning_rate=3e-4, critic_learning_rate=5e-4, n_steps=256, n_minibatches=4, n_epochs=5,
                      gamma=0.995, gae_lambda=0.95, clip_range=0.2, dual_clip=3.0, ent_coef=0.01,
                      d_model=64, attn_layers=2, hidden_size=128, recurrent=True,      # the networks
                      commander=True, commander_period=10,                             # the fleet commander
                      kl_bc_coef=0.1, opponent_refresh=10)                             # imitation anchor, league
pqn_settings = dict(learning_rate=2.5e-4, num_steps=128, epochs=4, minibatches=4, lam=0.85,
                    gamma=0.99, max_grad_norm=1.0,   # PQN's SMAX setting (Gallici et al. 2025): Q(lambda) on rollouts
                    mixer="vdn",                     # PQN-VDN: the fleet's Q is the sum of its ships' Q-values
                    exploration="softmax", temperature=0.1,   # Boltzmann: options by softmax(Q / 0.1);
                                                     #   "epsilon": epsilon-greedy (the paper's) - best option or random
                    exploration_initial_eps=1.0, exploration_final_eps=0.01, exploration_timesteps=300_000,
                    kl_bc_coef=0.5,                  # kickstarting: the pull towards the imitation clone, fading
                    d_model=64, attn_layers=2, hidden_size=128, recurrent=True,       # the same networks as MAPPO
                    commander=True, commander_period=10)                             # the same commander
save_interval = min(config['n_timesteps'] // 3, 250_000)   # a checkpoint (and team push) at least every 250,000

pd.Series(DEFAULT_CURRICULUM[STAGE], name=f"stage {STAGE}").to_frame()

## Environment

Launches `n_envs` headless Unity players - the real game. Both methods fight on the same players, each with its own battle log (`Monitor`) and its own curriculum stage. The observation has the same size in every stage (room for 9 ships a side and 5 circles), which is what lets one network play them all.

In [ ]:
battles = NavalEnv(stages="default", start_stage=STAGE, n_envs=config['n_envs'],
                   decision_period=config['decision_period'], sim_dt=config['sim_dt'],
                   unity_log_dir=os.path.join(run_dir, "unity_logs"))
env = Monitor(battles, log_dir)              # the MAPPO run's battles
env_pqn = Monitor(battles, pqn_log_dir)      # the PQN run's battles
print("battles in parallel:", battles.n_envs, "| token sizes", battles.dims)

## The two models

`open_run` loads each run's latest checkpoint with all its training memory - or, with no checkpoint yet, starts a new model from the imitation checkpoint (`models/imitation.pt`; the PQN one takes it as its Q-network, so with Boltzmann exploration it starts by playing like the Elite AI). It says when a run is on another stage: that run is then only evaluated here.

In [ ]:
imitation = os.path.join(models_dir, "imitation.pt")
model, mappo_here = open_run(env, STAGE, run_dir, log_dir, "mappo", storage=storage, force=FORCE_STAGE, **mappo_settings)
pqn, pqn_here = open_run(env_pqn, STAGE, pqn_run_dir, pqn_log_dir, "pqn", storage=pqn_storage, force=FORCE_STAGE,
                         imitation_path=imitation, **pqn_settings)
TRAIN_MAPPO, TRAIN_PQN = TRAIN_MAPPO and mappo_here, TRAIN_PQN and pqn_here
if storage and TRAIN_MAPPO:
    storage.lock()                   # stops here if a teammate is training this run right now
if pqn_storage and TRAIN_PQN:
    pqn_storage.lock()
for name, m, m_env, trained in (("MAPPO", model, env, TRAIN_MAPPO), ("PQN", pqn, env_pqn, TRAIN_PQN)):
    if trained:
        print(f"{name}: trains here, on stage {STAGE}")
    elif m is None:
        print(f"{name}: no model yet - start it in the Stage0 notebook")
    else:                            # each method climbs the curriculum on its own
        print(f"{name}: on stage {m_env.stage} - only evaluated here. To train it, open the Stage{m_env.stage} notebook "
              f"(or set FORCE_STAGE = True to train it on stage {STAGE}{' again' if m_env.stage > STAGE else ''})")
model.actor if model else None       # the captain (and the commander), as in the Week 10 notebooks

## Baselines

What the trained fleets have to beat on this stage: random legal actions, and a simple hand-written tactic. `pass_rate` is the stage's own goal (see the table at the top). Set `RUN_BASELINES = False` to skip them in later sessions.

In [ ]:
class Script:
    """Close in on the target, fire at will, launch torpedoes: a simple hand-written attack."""
    def predict(self, obs, deterministic=False):
        E, N = obs["alive"].shape
        actions = np.zeros((E, N, len(env.head_sizes)), dtype=np.int32)    # speed full, target auto, fire at will
        actions[..., 0] = np.where(obs["action_mask"][..., 9] > 0.5, 9, 0)    # move option 9 = close in
        actions[..., 4] = 1                                                   # torpedoes at the target
        return actions
SCRIPT_NAME = "scripted: close in, fire at will"

RUN_BASELINES = True
baselines = {}
if RUN_BASELINES:
    baselines = {"random": evaluate(None, env, n_battles=24, stage=STAGE),
                 SCRIPT_NAME: evaluate(Script(), env, n_battles=24, stage=STAGE)}
    display(pd.DataFrame(baselines).T[["stage", "battles", "win_rate", "pass_rate", "mean_reward"]])

## Train MAPPO

Training stops by itself as soon as the promotion requirement is met (unless `KEEP_GOING`), or when the stage cap in the table at the top is reached (the run then moves on, recorded as not passed), or after `n_timesteps`. `models/model_best.pt` always holds the checkpoint with the best rolling pass rate. The team folder gets a status update every 10,000 timesteps and a checkpoint every `save_interval`.

**Replays**: five times a session a battle is recorded and **shown right here, under the training table**, while training goes on (`runs_simple/<RUN_NAME>/videos/`: an MP4 to keep, and the GIF the notebook shows - an image plays in every notebook app, VS Code included; double-click it in the file browser). A replay shown while the cell runs is lost if the page is closed or reloaded meanwhile - **Replays from this stage** at the end shows every one again, from the files.

In the table: `win_rate` / `pass_rate` against the stage's own opponent.

In [ ]:
def callbacks_for(run_model, run_dir, models_dir, video_dir, run_storage):
    curriculum = CurriculumCallback(stop_on_promotion=not KEEP_GOING, fresh=FORCE_STAGE)
    callbacks = [SaveOnIntervalCallback(save_interval, models_dir),     # also refreshes model_latest.pt
                 curriculum,
                 BestModelCallback(models_dir),
                 VideoCallback(every=config['n_timesteps'] // 5, video_dir=video_dir, show=True)]   # replays appear below
    if run_storage:
        callbacks.append(TeamSyncCallback(run_storage, run_dir, push_every=save_interval, status_every=10_000))
    return callbacks, curriculum

if TRAIN_MAPPO:
    model.set_rival(pqn)             # cross-play: the PQN fleet joins MAPPO's league as "rival"
    callbacks, curriculum = callbacks_for(model, run_dir, models_dir, video_dir, storage)
    model.learn(total_timesteps=model.num_timesteps + config['n_timesteps'], callback=callbacks, log_interval=5)
    model.save(os.path.join(models_dir, "model_final"))
    model.save(os.path.join(models_dir, "model_latest"))

## Train PQN

The same battles, the same rewards, the same stopping rule - learned with PQN: every battle plays `num_steps` (128) decisions, then the Q-network learns from that rollout - Q(λ) targets, `epochs` (4) passes in `minibatches` (4) minibatches - and the rollout is thrown away. No replay buffer to fill and no target network to copy: it trains from the first rollout, with far less network work per timestep than a DQN sampling a replay buffer.

Cross-play: before each method trains, the other one's fleet - its champion, if it has one - joins its league as the **rival** (`set_rival`), so each learns to beat a differently trained fleet too. A *new champion* message means the fleet just beat its best self.

In the table: `q_loss` the squared TD error against the Q(λ) targets, `q_value` / `target_value` the Q-values and their targets (both should rise as the fleet starts winning), `epsilon` the commander's exploration rate, `rollout_decisions` the size of the last rollout.

In [ ]:
if TRAIN_PQN:
    pqn.set_rival(model)             # ... and the MAPPO fleet (just trained) joins PQN's
    callbacks, pqn_curriculum = callbacks_for(pqn, pqn_run_dir, pqn_models_dir, pqn_video_dir, pqn_storage)
    pqn.learn(total_timesteps=pqn.num_timesteps + config['n_timesteps'], callback=callbacks, log_interval=8)
    pqn.save(os.path.join(pqn_models_dir, "model_final"))
    pqn.save(os.path.join(pqn_models_dir, "model_latest"))

## Save and hand over

Saves both runs (`models/model_latest.pt`, where the next session continues) and pushes them to the team. **If you stop training early (Kernel > Interrupt), run this cell** - it saves what the interrupted session learned, so nothing is lost, and frees the runs for the next person.

In [ ]:
for name, m, m_env, m_storage, m_run_dir, trained in (("MAPPO", model, env, storage, run_dir, TRAIN_MAPPO),
                                                      ("PQN", pqn, env_pqn, pqn_storage, pqn_run_dir, TRAIN_PQN)):
    if m is None or not trained:
        continue
    m.save(os.path.join(m_run_dir, "models", "model_latest"))     # also after Kernel > Interrupt
    if m_storage:
        m_storage.push(m_run_dir, m, status=training_status(m, m_run_dir))
        m_storage.release()
    if m_env.stage > STAGE:
        unpassed = (curriculum if name == "MAPPO" else pqn_curriculum).moved_on_unpassed
        print(f"{name}: stage {STAGE} {'NOT passed - its timestep cap was reached' if unpassed else 'passed'} at "
              f"{m.num_timesteps:,} timesteps - now on '{m_env.stages[m_env.stage]['name']}': open the Stage{m_env.stage} notebook.")
    elif STAGE == len(m_env.stages) - 1 and (curriculum if name == "MAPPO" else pqn_curriculum).finished:
        capped = (curriculum if name == "MAPPO" else pqn_curriculum).capped_out
        print(f"{name}: training is complete ({m.num_timesteps:,} timesteps) - "
              f"{'its timestep cap was reached before the stopping rule was met' if capped else 'the stopping rule was met'}.")
    else:
        print(f"{name}: stage {STAGE} {'- the stopping rule decides when it is done' if STAGE == len(m_env.stages) - 1 else 'not passed yet'} "
              f"({m.num_timesteps:,} timesteps) - run this notebook again to continue.")

## Progress on this stage

The two methods side by side, against the timesteps each has spent **on this stage**: the pass rate (what promotion counts) and the reward.

In [ ]:
if storage:
    storage.show_status()
plot_comparison({"MAPPO": log_dir, "PQN": pqn_log_dir}, stage=STAGE)

**Reading the curriculum plots** (one per method): each colour is a stage, the y-axis its pass rate over the last 50 battles, and the dashed line the 70 % bar. A new colour starting means the run moved on - by passing, or at a stage's cap. A colour drawn **dotted** after that is not training on that stage: from stage 3 on, 10-15 % of the battles replay earlier stages (rehearsal) so their skills are not forgotten. The title says the furthest stage the run has reached - **that** stage's notebook is the one that trains it.

In [ ]:
plot_curriculum(log_dir, name="MAPPO")                          # pass rate per stage, one colour each
plot_curriculum(pqn_log_dir, name="PQN")
plot_reward_terms(log_dir, stage=STAGE, name="MAPPO")           # what the fleet is paid for on this stage
plot_reward_terms(pqn_log_dir, stage=STAGE, name="PQN")

In [ ]:
plot_progress(log_dir)                       # PPO's diagnostics
plot_progress(pqn_log_dir)                   # PQN's diagnostics

## Evaluation: MAPPO vs PQN

Fresh battles on this stage against its own opponent, for every model: each plays the way it trains - MAPPO samples its policy, PQN samples softmax(Q / 0.1) (with `exploration="epsilon"`: its best action, 2% random). The best checkpoints are evaluated too.

In [ ]:
results = dict(baselines)
for name, cls, m, mdir in (("MAPPO", MAPPO, model, models_dir), ("PQN", FleetPQN, pqn, pqn_models_dir)):
    if m is None:
        print(f"{name}: no model yet - train it from Stage 0")
        continue
    m_env = env if cls is MAPPO else env_pqn
    results[f"{name} (latest)"] = evaluate(m, m_env, n_battles=40, stage=STAGE)
    best_path = os.path.join(mdir, "model_best.pt")
    if os.path.exists(best_path):
        stage_now = m_env.stage
        best = cls.load(best_path, env=m_env)          # (loading moves that run's stage to the one it was saved on)
        results[f"{name} (best)"] = evaluate(best, m_env, n_battles=40, stage=STAGE)
        m_env.set_stage(stage_now)
columns = ["battles", "win_rate", "pass_rate", "win_rate_stderr", "mean_reward", "enemy_ships_sunk", "own_ships_lost",
           "points", "enemy_points"]
table = pd.DataFrame(results).T
table[[c for c in columns if c in table]]

## Every stage so far

How each method did on every stage of the curriculum, from its battle log: the timesteps it spent there until it passed, its pass rate over its last 50 qualifying battles there, and whether it passed.

In [ ]:
compare_stages({"MAPPO": log_dir, "PQN": pqn_log_dir}, env.stages)

## Put the fleets into your Unity game

Exports each run's best checkpoint (or its latest, if that has reached a later stage) as a policy file the game runs in plain C# (`Assets/Scripts/RL/RLPolicy.cs`): MAPPO's as `naval_policy.bin`, PQN's as `naval_policy_pqn.bin`. From the repository they go straight into `Assets/StreamingAssets/RL/`; elsewhere (SageMaker) into the run folders - download them (right-click > Download) and copy them there.

Then in Unity: **Play** > **TRAINING STAGES**; at the bottom pick **TRAINED FLEET: MAPPO** or **PQN**, and **WATCH MAPPO** / **WATCH PQN** on any stage, or **MAPPO v PQN** to put the two fleets against each other (or on the setup screen: **LEARNED = MAPPO / PQN** with **YOUR FLEET: LEARNED** / **ENEMY AI: LEARNED**). In battle, the banner under the score names the policy flying each side. From a computer later: `python export_simple.py` and `python export_simple.py --run-name fleet_pqn`.

In [ ]:
from simple_mappo.export import GAME_POLICY, GAME_POLICY_PQN, export_policy, pick_checkpoint
in_repo = os.path.isdir(os.path.join("..", "Assets", "StreamingAssets"))
for mdir, rdir, game_file in ((models_dir, run_dir, GAME_POLICY), (pqn_models_dir, pqn_run_dir, GAME_POLICY_PQN)):
    checkpoint = pick_checkpoint(mdir)        # model_best.pt, or model_latest.pt if it reached a later stage
    if checkpoint is None or checkpoint.endswith("imitation.pt") and mdir == pqn_models_dir:
        continue
    target = game_file if in_repo else os.path.join(rdir, os.path.basename(game_file))
    print("exported", checkpoint, "->", os.path.abspath(export_policy(checkpoint, target)))

## Replays from this stage

**Every** replay recorded on this stage, MAPPO's then PQN's, oldest first: the ones from this session (also those whose live display above was lost when the page was closed), from earlier sessions, and - with team checkpoints - the ones a teammate or a training job recorded, downloaded from the team folder first.

A run records replays only on the stage it **trains** on. A run that is on another stage (each method climbs the curriculum at its own pace) is only evaluated here - so for it, one battle on this stage is recorded now (`..._evaluated`, once per checkpoint), and both methods always have a replay of this stage to compare. Each is embedded in the notebook as an animated GIF - an image plays in every notebook app (JupyterLab, SageMaker Studio, VS Code, Cursor; VS Code blocks embedded videos) and stays in the notebook when it is reopened or downloaded. An older replay with only an MP4 gets its GIF made here. `ALL_STAGES = True` shows every replay of both runs, from every stage.

They are a 2D map drawn from the game's data (the training player has no graphics); to see a fleet in the real game's graphics, export it (above) and use **WATCH MAPPO** / **WATCH PQN** in Unity.

In [ ]:
ALL_STAGES = False                   # True: every replay of both runs, from every stage
for replay_storage, replay_run_dir in ((storage, run_dir), (pqn_storage, pqn_run_dir)):
    if replay_storage:
        print(f"{replay_storage.pull_videos(replay_run_dir)} new replay(s) from {replay_storage.location}")
# a run that is on another stage did not train here, so it recorded nothing on this stage: record one battle
for name, m, m_env, m_video_dir, trained in (("MAPPO", model, env, video_dir, TRAIN_MAPPO),
                                             ("PQN", pqn, env_pqn, pqn_video_dir, TRAIN_PQN)):
    stem = f"battle_{m.num_timesteps:09d}_stage{STAGE}" if m is not None else ""
    if m is not None and not trained and not find_videos(m_video_dir, prefix=stem):
        print(f"{name} is on stage {m_env.stage} ('{m_env.stages[m_env.stage]['name']}'), so it did not train here - "
              f"recording one battle of it on stage {STAGE}:")
        record_battle(m, m_env, os.path.join(m_video_dir, f"{stem}_evaluated{video_ext()}"), stage=STAGE)
show_videos({"MAPPO": video_dir, "PQN": pqn_video_dir}, stage=None if ALL_STAGES else STAGE)

In [ ]:
battles.close()